# Fetal brain US — stratified grouped 5-fold CV (full pipeline)
Runtime → **T4 GPU**, then *Run all*. Only cell 2 needs your input (the dataset zip in Drive).
Pipeline: unzip → design classes from real case counts → 5 stratified grouped folds → train (regularised) →
evaluate (CIs, real baseline) → frozen-feature probe → back up to Drive.

In [1]:
# 1. GPU + Drive
import tensorflow as tf, subprocess, sys, os, glob, zipfile, shutil, json
print('GPU:', tf.config.list_physical_devices('GPU'))
from google.colab import drive
drive.mount('/content/drive')

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mounted at /content/drive


In [2]:
# 2. Dataset zip in Drive  (set ZIP, or leave it to search)
ZIP = '/content/drive/MyDrive/FetalBrain_XAI_Workspace/02_Original_Dataset/Fetal Brain Abnormalities Ultrasound.v1i.multiclass.zip'
if not os.path.exists(ZIP):   # fall back to a search, ignoring model backups
    cands = glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
    print('zip files found:'); [print('  ', c) for c in cands]
    key = [c for c in cands if 'backup' not in c.lower()
           and any(k in os.path.basename(c).lower() for k in ('fetal', 'foetal', 'brain', 'ultrasound', 'dataset', 'roboflow'))]
    assert len(key) == 1, 'set ZIP above to one of the paths listed'
    ZIP = key[0]
print('using', ZIP)
shutil.rmtree('/content/ds_raw', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content/ds_raw')
# the export root is the folder that contains train/ (with _classes.csv or class folders)
RAW = next(os.path.dirname(p) for p in glob.glob('/content/ds_raw/**/train', recursive=True))
print('RAW =', RAW, {s: len(os.listdir(f'{RAW}/{s}')) for s in ('train', 'valid', 'test') if os.path.isdir(f'{RAW}/{s}')})

In [ ]:
# 3. Code from GitHub (always the latest)
if os.path.exists('/content/repo'):
    subprocess.run(['git', '-C', '/content/repo', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/MUKUL-PRASAD-SIGH/XAI', '/content/repo'], check=True)
S = '/content/repo/scripts'
def run(*a):
    r = subprocess.run([sys.executable, *map(str, a)], capture_output=True, text=True)
    print(r.stdout[-7000:]); print(r.stderr[-1500:] if r.returncode else '')
    assert r.returncode == 0, 'step failed'

In [ ]:
# 4. Audit (leakage / duplicates / sources) — read it, it explains the dataset
run(f'{S}/01_dataset_audit.py', '--src', RAW, '--out', '/content/audit')

In [ ]:
# 5. Design the classes from the REAL number of independent cases per diagnosis
run(f'{S}/02c_design_classes.py', '--src', RAW, '--out', '/content/class_map_auto.json', '--min-groups', 5)

In [ ]:
# 6. Stratified grouped 5-fold dataset
CV = '/content/dataset_cv'
shutil.rmtree(CV, ignore_errors=True)
run(f'{S}/02b_make_cv_folds.py', '--src', RAW, '--out', CV, '--folds', 5,
    '--class-map', '/content/class_map_auto.json', '--min-groups', 5, '--link')

In [ ]:
# 7. Train the 5 folds ONE AT A TIME: live progress, resumable, each fold backed up to Drive
# RUN picks the checkpoint folder. A NEW name trains all 5 folds from scratch into models_cv_<RUN>
# (local and Drive); the baseline in cv_run/models_cv is never read or overwritten. Re-running the
# SAME name resumes that run after a disconnect. RUN = '' reuses the baseline checkpoints (no training).
RUN = 'retrain_v2'
import subprocess, sys, os, shutil
SFX = f'_{RUN}' if RUN else ''
MODELS, RESULTS, PROBE = f'/content/models_cv{SFX}', f'/content/results_cv{SFX}', f'/content/results_probe{SFX}'
BK = f'/content/drive/MyDrive/FetalBrain_XAI_Workspace/cv_run/models_cv{SFX}'
print('run:', RUN or 'baseline', '| checkpoints:', MODELS, '| Drive backup:', BK)
os.makedirs(BK, exist_ok=True); os.makedirs(MODELS, exist_ok=True)
for f in os.listdir(BK):                      # restore folds of THIS run finished in an earlier session
    if not os.path.exists(f'{MODELS}/{f}'):
        shutil.copy(f'{BK}/{f}', MODELS)
n_folds = len([d for d in os.listdir(CV) if d.startswith('fold_')])
for k in range(n_folds):
    ck = f'{MODELS}/best_model_xception_fold_{k}.keras'
    if os.path.exists(ck):
        print(f'fold {k}: already trained in run {RUN or "baseline"!r}, skipping'); continue
    print(f'\n########## training fold {k + 1}/{n_folds} ##########')
    p = subprocess.Popen([sys.executable, '-u', f'{S}/03_train.py', '--cv', CV, '--model', 'xception',
                          '--out', MODELS, '--mixed-precision', '--only-folds', str(k)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        if 'ETA' in line or '\r' in line:      # progress-bar noise
            continue
        print(line, end=''); tail.append(line); tail = tail[-60:]
    p.wait()
    if p.returncode or not os.path.exists(ck):
        print('\n--- LAST OUTPUT ---'); print(''.join(tail[-40:]))
        raise RuntimeError(f'training fold {k} failed (return code {p.returncode})')
    for f in os.listdir(MODELS):              # back up after every fold
        if f.startswith(f'best_model_xception_fold_{k}'):
            shutil.copy(f'{MODELS}/{f}', BK)
    print(f'fold {k} done and backed up')
print('\nCheckpoints:', sorted(f for f in os.listdir(MODELS) if f.endswith('.keras')))


In [ ]:
# 8. Evaluate pooled CV + frozen-feature probe on the same folds
# count only Xception folds: cell 8d also writes cnn/scnn checkpoints into the same folder
ckpts = [f for f in os.listdir(MODELS) if f.startswith('best_model_xception_fold_') and f.endswith('.keras')]
assert len(ckpts) == n_folds, f'only {len(ckpts)}/{n_folds} xception fold checkpoints exist - rerun cell 7'
run(f'{S}/04_evaluate.py', '--cv', CV, '--models', MODELS, '--out', RESULTS, '--tta')
run(f'{S}/13_cv_probe.py', '--cv', CV, '--out', PROBE)
print(open(f'{RESULTS}/results_table.md').read())

In [ ]:
# 8b. Helper: train one model type over all folds, one at a time, resumable, backed up to Drive
def train_folds(model, out_dir, bk, extra=(), size_args=()):
    os.makedirs(bk, exist_ok=True); os.makedirs(out_dir, exist_ok=True)
    for f in os.listdir(bk):                       # restore folds finished in an earlier session
        if f.startswith(f'best_model_{model}_') and not os.path.exists(f'{out_dir}/{f}'):
            shutil.copy(f'{bk}/{f}', out_dir)
    n = len([d for d in os.listdir(CV) if d.startswith('fold_')])
    for k in range(n):
        ck = f'{out_dir}/best_model_{model}_fold_{k}.keras'
        if os.path.exists(ck):
            print(f'{model} fold {k}: already trained, skipping'); continue
        print(f'\n########## {model}: fold {k + 1}/{n} ##########')
        p = subprocess.Popen([sys.executable, '-u', f'{S}/03_train.py', '--cv', CV, '--model', model,
                              '--out', out_dir, '--mixed-precision', '--only-folds', str(k), *map(str, extra)],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        tail = []
        for line in p.stdout:
            if 'ETA' in line or '\r' in line:
                continue
            print(line, end=''); tail.append(line); tail = tail[-60:]
        p.wait()
        if p.returncode or not os.path.exists(ck):
            print('\n--- LAST OUTPUT ---'); print(''.join(tail[-40:]))
            raise RuntimeError(f'{model} fold {k} failed (return code {p.returncode})')
        for f in os.listdir(out_dir):
            if f.startswith(f'best_model_{model}_fold_{k}'):
                shutil.copy(f'{out_dir}/{f}', bk)
        print(f'{model} fold {k} done and backed up')

In [ ]:
# 8c. Audit the Xception run you just evaluated (read-only: no training, existing outputs untouched)
import time
AUD = f'/content/audit_cv{SFX}_{time.strftime("%Y%m%d_%H%M%S")}'
run(f'{S}/16_audit_cv_results.py', '--cv', CV, '--results', RESULTS,
    '--models', MODELS, '--probe', PROBE, '--out', AUD)
print(open(f'{AUD}/audit_report.md').read())

In [ ]:
# 8d. CNN and S-CNN baselines. They were skipped before because no checkpoints existed; this trains them
# (5 folds each, ~224px from scratch) and re-evaluates all three models into a NEW results folder.
BK_ALL = BK                                   # same run folder as cell 7
for m in ('cnn', 'scnn'):
    train_folds(m, MODELS, BK_ALL)
run(f'{S}/04_evaluate.py', '--cv', CV, '--models', MODELS, '--out', f'/content/results_cv_all{SFX}', '--tta')
print(open(f'/content/results_cv_all{SFX}/results_table.md').read())

In [ ]:
import time
# 9. NEW DATA: anatomy pretraining on the Fetal-Head sets (Trans-cerebellum, Trans-ventricular, Diverse),
# then the SAME 5 folds with the pretrained backbone. The Fetal-Head images only carry Brain/CSP/LV masks,
# so they are used for the pretext task only - no diagnosis label is invented. Images that duplicate a
# Roboflow image are excluded so the backbone never sees a diagnosis test scan.
NEW = '/content/source_zips'; os.makedirs(NEW, exist_ok=True)
for name in ('Trans-cerebellum.zip', 'Trans-ventricular.zip', 'Diverse Fetal Head Images.zip',
             'ahuteam FUSEP main labels.zip'):
    hits = [h for h in glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True) if 'backup' not in h.lower()]
    if hits:
        shutil.copy(hits[0], NEW); print('found', hits[0])
    else:
        print('NOT FOUND in Drive:', name)
shutil.copy(ZIP, NEW)                                  # Roboflow zip, needed for the duplicate check
run(f'{S}/14_build_unified_dataset.py', '--root', NEW)
BUILD = sorted(glob.glob('/content/repo/datasets/final_build_*'))[-1]
print('BUILD =', BUILD)
PRE = '/content/anatomy_pretrain'
run(f'{S}/15_pretrain_anatomy.py', '--build-dir', BUILD, '--out', PRE, '--mixed-precision')
print(open(f'{PRE}/pretrain_result.json').read())
BK_ANAT = '/content/drive/MyDrive/FetalBrain_XAI_Workspace/cv_run/models_cv_anatomy' + SFX
train_folds('xception', f'/content/models_cv_anat{SFX}', BK_ANAT,
            extra=['--init-backbone', f'{PRE}/xception_anatomy_backbone.weights.h5'])
run(f'{S}/04_evaluate.py', '--cv', CV, '--models', f'/content/models_cv_anat{SFX}', '--out', f'/content/results_cv_anat{SFX}', '--tta')
print(open(f'/content/results_cv_anat{SFX}/results_table.md').read())
AUD2 = f'/content/audit_cv_anat{SFX}_{time.strftime("%Y%m%d_%H%M%S")}'
run(f'{S}/16_audit_cv_results.py', '--cv', CV, '--results', f'/content/results_cv_anat{SFX}',
    '--models', f'/content/models_cv_anat{SFX}', '--out', AUD2)
print(open(f'{AUD2}/audit_report.md').read())

In [ ]:
# 9. Optional second pass: also merge classes the model itself confuses, then redo 6-8
# run(f'{S}/02c_design_classes.py', '--src', RAW, '--out', '/content/class_map_auto2.json',
#     '--predictions', '/content/results_cv/predictions_xception.csv')

In [ ]:
# 10. Back up everything to Drive
OUT = '/content/drive/MyDrive/FetalBrain_XAI_Workspace/cv_run'
os.makedirs(OUT, exist_ok=True)
for d in (MODELS, RESULTS, PROBE, '/content/audit'):   # run-tagged names: the baseline folders are not overwritten
    if os.path.isdir(d):
        shutil.copytree(d, f'{OUT}/{os.path.basename(d)}', dirs_exist_ok=True)
for f in ('class_map_auto.json', 'class_map_auto.report.txt'):
    shutil.copy(f'/content/{f}', OUT)
shutil.copy(f'{CV}/cv_report.txt', OUT); shutil.copy(f'{CV}/cv_manifest.csv', OUT)
print('saved to', OUT)